# Lab 1.5: Batch Processing Strategies

You're the architect for a document processing platform. Your team is excited about the Message Batches API's 50% cost savings and wants to use it everywhere. In this lab you'll discover why that's a disaster for blocking workflows, then build the decision logic, submission math, and failure recovery that a production system actually needs.

**Why this lab is different:** Batch API calls are expensive and slow (up to 24 hours). Instead of calling the actual API, this lab uses simulation and decision exercises so you can focus on the architectural reasoning the exam tests.

**What you'll learn:**
- Why using batch for blocking workflows destroys developer velocity
- How to route workflows to sync vs batch API
- SLA math for calculating batch submission frequency
- Building batch request objects with `custom_id`
- Failure handling: identify, chunk, resubmit
- Cost savings analysis

## Setup

In [ ]:
import anthropic
import json
import time
import random

client = anthropic.Anthropic()  # uses ANTHROPIC_API_KEY env var
MODEL = "claude-sonnet-4-20250514"

---
## Phase 1: THE WRONG WAY -- Batch Everything

Your team has proposed using the Batch API for every workflow in the platform. The pitch: "50% cost savings across the board!" Let's see what happens.

In [ ]:
# The team's proposed architecture: batch everything
PROPOSED_WORKFLOWS = [
    {"name": "Pre-merge code review",     "blocking": True,  "proposed_api": "batch", "typical_volume": "50/day"},
    {"name": "Nightly security audit",     "blocking": False, "proposed_api": "batch", "typical_volume": "1/day"},
    {"name": "PR comment generation",      "blocking": True,  "proposed_api": "batch", "typical_volume": "80/day"},
    {"name": "Weekly tech debt report",    "blocking": False, "proposed_api": "batch", "typical_volume": "1/week"},
    {"name": "Real-time chat support",     "blocking": True,  "proposed_api": "batch", "typical_volume": "200/day"},
]

print("=== PROPOSED ARCHITECTURE ===")
print(f"{'Workflow':<30} {'Blocking?':<12} {'Proposed API':<15} {'Volume'}")
print("-" * 75)
for w in PROPOSED_WORKFLOWS:
    print(f"{w['name']:<30} {str(w['blocking']):<12} {w['proposed_api']:<15} {w['typical_volume']}")

print(f"\nTeam's argument: 'We save 50% on ALL of these!'")
print(f"Sounds great... right?")

In [ ]:
# Simulate: What happens when batch takes 18 hours for a blocking workflow?

def simulate_batch_day(workflows, seed=42):
    """Simulate one day where batch processing takes variable time."""
    random.seed(seed)
    
    print("=== SIMULATING A TYPICAL MONDAY ===")
    print()
    
    blocked_hours = 0
    blocked_workflows = []
    
    for w in workflows:
        # Batch processing time is unpredictable: 1-24 hours
        batch_hours = round(random.uniform(1, 24), 1)
        
        if w["blocking"]:
            print(f"  {w['name']}:")
            print(f"    Developer submits PR at 9:00 AM")
            print(f"    Batch review submitted... processing...")
            print(f"    Batch completes after {batch_hours} hours")
            
            if batch_hours > 1:
                finish_time = 9 + batch_hours
                if finish_time > 17:  # After 5 PM
                    print(f"    Result arrives at {finish_time:.1f}:00 -- AFTER WORK HOURS")
                    print(f"    Developer blocked ALL DAY. PR doesn't merge until tomorrow.")
                    blocked_hours += 8
                else:
                    h = int(finish_time)
                    m = int((finish_time - h) * 60)
                    print(f"    Result arrives at {h}:{m:02d} PM")
                    print(f"    Developer blocked for {batch_hours:.1f} hours")
                    blocked_hours += batch_hours
                blocked_workflows.append(w["name"])
            print()
    
    print("=" * 50)
    print(f"DAMAGE REPORT:")
    print(f"  Workflows blocked: {len(blocked_workflows)}")
    print(f"  Total developer-hours lost: {blocked_hours:.1f}")
    print(f"  Blocked workflows: {blocked_workflows}")
    print()
    print(f"  And this is just ONE day. Multiply by your team size.")
    print(f"  The 50% API cost savings is meaningless compared to")
    print(f"  the developer time wasted waiting.")

simulate_batch_day(PROPOSED_WORKFLOWS)

In [ ]:
# The argument you'll hear (and the exam tests):
# "Batch usually finishes in a few hours, so it's fine for pre-merge checks."

print('=== THE "USUALLY FAST" TRAP ===')
print()
print('Claim: "Batch usually finishes in 2-3 hours"')
print()

# Simulate 20 batch submissions
random.seed(123)
times = [round(random.uniform(0.5, 24), 1) for _ in range(20)]
times.sort()

print(f"Actual completion times over 20 submissions (hours):")
for i, t in enumerate(times):
    marker = " <-- would block a full workday" if t > 8 else ""
    print(f"  Submission {i+1:2d}: {t:5.1f}h{marker}")

fast = sum(1 for t in times if t < 3)
slow = sum(1 for t in times if t > 8)
print(f"\n  Under 3 hours: {fast}/20 ({fast/20:.0%})")
print(f"  Over 8 hours:  {slow}/20 ({slow/20:.0%})")
print(f"\n  Even if it's fast 75% of the time, the other 25% destroys your sprint.")
print(f'  "Usually fast" is NOT an SLA. The Batch API has no latency guarantee.')

### Phase 1 Takeaway

The Batch API has **no latency SLA**. Processing can take anywhere from minutes to 24 hours. Using it for blocking workflows means unpredictable developer wait times. The 50% cost savings is irrelevant if developers are idle for hours waiting on results.

**Exam trap:** "Batch usually finishes quickly" is always wrong for blocking workflows. The exam will present this exact reasoning as a distractor.

---
## Phase 2: THE RIGHT WAY -- Route Correctly

The correct architecture routes each workflow to the right API based on whether it blocks human work.

In [ ]:
def choose_api(workflow):
    """Route a workflow to sync or batch API.
    
    Rule: If a human or process is waiting for the result (blocking=True),
    use synchronous API. Otherwise, use batch for 50% cost savings.
    """
    if workflow["blocking"]:
        return "sync"
    else:
        return "batch"

# Apply correct routing
print("=== CORRECT ARCHITECTURE ===")
print(f"{'Workflow':<30} {'Blocking?':<12} {'Correct API':<15} {'Reason'}")
print("-" * 85)
for w in PROPOSED_WORKFLOWS:
    api = choose_api(w)
    reason = "Humans waiting -- need immediate results" if api == "sync" else "No one waiting -- save 50%"
    print(f"{w['name']:<30} {str(w['blocking']):<12} {api:<15} {reason}")

In [ ]:
# === SUBMISSION FREQUENCY: SLA MATH ===

def calculate_submission_interval(sla_hours, batch_max_hours=24):
    """Calculate how often to submit batches to meet an SLA.
    
    If your SLA says results within `sla_hours`, and batch can take
    up to `batch_max_hours`, the maximum interval between submissions is:
    
        max_interval = sla_hours - batch_max_hours
    
    If this is <= 0, batch API CANNOT meet the SLA.
    """
    buffer = sla_hours - batch_max_hours
    
    if buffer <= 0:
        return None  # Batch cannot meet this SLA
    
    return buffer

# Examples
test_slas = [72, 48, 30, 24, 12, 8]

print("=== SUBMISSION FREQUENCY CALCULATOR ===")
print(f"{'SLA (hours)':<15} {'Max Interval':<20} {'Feasible?'}")
print("-" * 50)
for sla in test_slas:
    interval = calculate_submission_interval(sla)
    if interval is None:
        print(f"{sla:<15} {'N/A':<20} NO -- batch cannot guarantee this SLA")
    else:
        print(f"{sla:<15} {f'Every {interval}h':<20} YES")

print(f"\nKey insight: SLA of exactly 24h means 0 buffer -- batch can't guarantee it.")
print(f"You need SLA > 24h to use batch. The tighter the SLA, the more often you submit.")

In [ ]:
# === BUILDING BATCH REQUEST OBJECTS ===

def build_batch_requests(documents):
    """Build batch request objects with custom_id for correlation.
    
    custom_id is your key to matching responses back to requests
    and identifying which specific documents failed.
    """
    requests = []
    for doc in documents:
        request = {
            "custom_id": f"doc-{doc['id']}",  # YOUR correlation key
            "params": {
                "model": MODEL,
                "max_tokens": 2000,
                "messages": [
                    {
                        "role": "user",
                        "content": f"Analyze this document for security issues:\n\n{doc['content']}"
                    }
                ]
            }
        }
        requests.append(request)
    return requests

# Example documents
sample_docs = [
    {"id": "SEC-001", "content": "Authentication module with OAuth2 flow..."},
    {"id": "SEC-002", "content": "Payment processing with PCI compliance..."},
    {"id": "SEC-003", "content": "User data export GDPR handler..."},
    {"id": "SEC-004", "content": "Admin panel with role-based access..."},
    {"id": "SEC-005", "content": "API rate limiting middleware..."},
]

batch_requests = build_batch_requests(sample_docs)

print("=== BATCH REQUEST OBJECTS ===")
print(f"Total requests: {len(batch_requests)}\n")
for req in batch_requests:
    print(f"  custom_id: {req['custom_id']}")
    content_preview = req['params']['messages'][0]['content'][:50]
    print(f"  content:   {content_preview}...")
    print()

In [ ]:
# === FAILURE HANDLING ===

def simulate_batch_results(requests, failure_rate=0.15, seed=42):
    """Simulate batch results with some failures."""
    random.seed(seed)
    results = []
    for req in requests:
        if random.random() < failure_rate:
            # Simulate different failure types
            error_type = random.choice(["overloaded", "context_too_long", "timeout"])
            results.append({
                "custom_id": req["custom_id"],
                "status": "errored",
                "error": {"type": error_type, "message": f"Request failed: {error_type}"}
            })
        else:
            results.append({
                "custom_id": req["custom_id"],
                "status": "succeeded",
                "result": {"content": f"Analysis complete for {req['custom_id']}"}
            })
    return results


def handle_batch_failures(results, original_requests):
    """Identify failed requests and prepare resubmission.
    
    Key principle: Use custom_id to identify failures,
    then resubmit ONLY the failed ones (not the entire batch).
    """
    succeeded = [r for r in results if r["status"] == "succeeded"]
    failed = [r for r in results if r["status"] == "errored"]
    
    print(f"=== BATCH RESULTS ===")
    print(f"  Succeeded: {len(succeeded)}")
    print(f"  Failed:    {len(failed)}")
    print()
    
    if not failed:
        print("  All requests succeeded!")
        return [], []
    
    # Map custom_ids back to original requests
    request_lookup = {req["custom_id"]: req for req in original_requests}
    
    resubmit = []
    needs_chunking = []
    
    print("  Failed requests:")
    for f in failed:
        error_type = f["error"]["type"]
        original = request_lookup[f["custom_id"]]
        print(f"    {f['custom_id']}: {error_type}")
        
        if error_type == "context_too_long":
            # Don't just retry -- chunk the document
            print(f"      -> Needs chunking before resubmit")
            needs_chunking.append(original)
        else:
            # Transient error -- safe to retry as-is
            print(f"      -> Resubmit as-is")
            resubmit.append(original)
    
    print(f"\n  Action plan:")
    print(f"    Resubmit directly: {len(resubmit)}")
    print(f"    Chunk then resubmit: {len(needs_chunking)}")
    print(f"    Already succeeded (do NOT resubmit): {len(succeeded)}")
    
    return resubmit, needs_chunking


# Run simulation with 15% failure rate
results = simulate_batch_results(batch_requests, failure_rate=0.15)
resubmit, chunk = handle_batch_failures(results, batch_requests)

In [ ]:
# === COST SAVINGS ANALYSIS ===

def cost_analysis(workflows):
    """Calculate cost savings from correct sync/batch routing."""
    
    # Approximate costs (illustrative)
    COST_PER_CALL_SYNC = 0.015   # $0.015 per API call (sync)
    COST_PER_CALL_BATCH = 0.0075  # $0.0075 per API call (batch = 50% of sync)
    DEVELOPER_HOURLY_RATE = 75    # $/hr
    
    volume_map = {
        "50/day": 50 * 22,   # monthly (22 work days)
        "80/day": 80 * 22,
        "200/day": 200 * 22,
        "1/day": 22,
        "1/week": 4,
    }
    
    print("=== MONTHLY COST ANALYSIS ===")
    print(f"{'Workflow':<30} {'API':<8} {'Volume':<10} {'API Cost':<12} {'Wait Cost':<12} {'Total'}")
    print("-" * 90)
    
    total_api_cost = 0
    total_wait_cost = 0
    all_batch_api_cost = 0  # What it would cost if everything were batch
    all_batch_wait_cost = 0
    
    for w in workflows:
        monthly_vol = volume_map[w["typical_volume"]]
        api = choose_api(w)
        
        if api == "sync":
            api_cost = monthly_vol * COST_PER_CALL_SYNC
            wait_cost = 0  # Sync returns in seconds
        else:
            api_cost = monthly_vol * COST_PER_CALL_BATCH
            wait_cost = 0  # Non-blocking, no one waiting
        
        # What if we used batch for everything (the wrong way)?
        batch_api_cost = monthly_vol * COST_PER_CALL_BATCH
        if w["blocking"]:
            # Average 6 hours blocked per call, but only for blocking workflows
            avg_wait_hours = 6
            batch_wait_cost = monthly_vol * avg_wait_hours * DEVELOPER_HOURLY_RATE
        else:
            batch_wait_cost = 0
        
        total_api_cost += api_cost
        total_wait_cost += wait_cost
        all_batch_api_cost += batch_api_cost
        all_batch_wait_cost += batch_wait_cost
        
        print(f"{w['name']:<30} {api:<8} {monthly_vol:<10} ${api_cost:<11.2f} ${wait_cost:<11.2f} ${api_cost + wait_cost:.2f}")
    
    print("-" * 90)
    correct_total = total_api_cost + total_wait_cost
    wrong_total = all_batch_api_cost + all_batch_wait_cost
    
    print(f"\n{'CORRECT (sync + batch):':<35} ${correct_total:>10,.2f}/month")
    print(f"{'WRONG (all batch):':<35} ${all_batch_api_cost:>10,.2f}/month in API costs")
    print(f"{'':35} ${all_batch_wait_cost:>10,.2f}/month in developer wait time")
    print(f"{'':35} ${wrong_total:>10,.2f}/month TOTAL")
    print(f"\n  The 'batch everything' approach saves ${total_api_cost - all_batch_api_cost:,.2f} on API costs...")
    print(f"  ...but costs ${all_batch_wait_cost:,.2f} in blocked developer time.")
    print(f"\n  Net loss from batch-everything: ${wrong_total - correct_total:,.2f}/month")

cost_analysis(PROPOSED_WORKFLOWS)

### Phase 2 Summary

The correct architecture:
1. **Route by blocking status** -- sync for blocking, batch for non-blocking
2. **SLA math** -- `max_interval = sla_hours - 24`. If zero or negative, use sync.
3. **Use `custom_id`** -- correlate requests to responses, identify failures
4. **Smart failure recovery** -- resubmit only failed, chunk oversized, don't retry the whole batch
5. **Cost savings are real** -- but only for non-blocking workflows

---
## Phase 3: YOUR TURN -- Classify and Calculate

You've been given 8 new workflows. For each one, decide: sync or batch? Then calculate the submission frequency for a specific SLA, and design failure recovery.

In [ ]:
# === YOUR WORKFLOWS ===

YOUR_WORKFLOWS = [
    {"id": 1, "name": "Customer support auto-reply",       "blocking": True,  "latency_need": "seconds"},
    {"id": 2, "name": "Monthly compliance report",         "blocking": False, "latency_need": "days"},
    {"id": 3, "name": "Deployment approval gate",          "blocking": True,  "latency_need": "minutes"},
    {"id": 4, "name": "Quarterly training data labeling",  "blocking": False, "latency_need": "weeks"},
    {"id": 5, "name": "Incident triage classification",    "blocking": True,  "latency_need": "seconds"},
    {"id": 6, "name": "Daily test coverage summary",       "blocking": False, "latency_need": "overnight"},
    {"id": 7, "name": "Merge conflict resolution helper",  "blocking": True,  "latency_need": "minutes"},
    {"id": 8, "name": "Weekly documentation freshness check","blocking": False, "latency_need": "days"},
]

print("=== YOUR WORKFLOWS ===")
for w in YOUR_WORKFLOWS:
    print(f"  {w['id']}. {w['name']:<45} blocking={w['blocking']}, latency={w['latency_need']}")

print("\nYour tasks:")
print("  1. Classify each as sync or batch")
print("  2. For a 30-hour SLA, calculate submission frequency for batch workflows")
print("  3. Design failure recovery for a batch with 5% failure rate")

In [ ]:
# ============================================================
# TASK 1: Classify each workflow
# Fill in "sync" or "batch" for each workflow ID
# ============================================================

your_classifications = {
    1: "???",  # Customer support auto-reply
    2: "???",  # Monthly compliance report
    3: "???",  # Deployment approval gate
    4: "???",  # Quarterly training data labeling
    5: "???",  # Incident triage classification
    6: "???",  # Daily test coverage summary
    7: "???",  # Merge conflict resolution helper
    8: "???",  # Weekly documentation freshness check
}

# ============================================================
# TASK 2: Calculate submission frequency for 30-hour SLA
# ============================================================

sla_hours = 30
your_submission_interval = "???"  # How many hours between batch submissions?

# ============================================================
# TASK 3: Failure recovery design
# Given a batch of 200 documents with 5% failure rate:
# - How many failures expected?
# - What do you resubmit?
# - What do you NOT resubmit?
# ============================================================

your_failure_plan = {
    "expected_failures": "???",         # Number
    "resubmit_what": "???",             # Describe what gets resubmitted
    "do_not_resubmit": "???",           # Describe what you skip
    "how_to_identify_failures": "???",  # How do you know which ones failed?
}

In [ ]:
# === CHECKER ===

CORRECT_CLASSIFICATIONS = {
    1: "sync",   # Customer support -- users waiting
    2: "batch",  # Monthly report -- no one waiting
    3: "sync",   # Deployment gate -- pipeline blocked
    4: "batch",  # Quarterly labeling -- flexible timeline
    5: "sync",   # Incident triage -- urgent, someone waiting
    6: "batch",  # Daily summary -- overnight is fine
    7: "sync",   # Merge conflict -- developer waiting
    8: "batch",  # Weekly doc check -- days of slack
}

print("=== TASK 1: Classification Check ===")
all_correct = True
for wid, correct in CORRECT_CLASSIFICATIONS.items():
    yours = your_classifications.get(wid, "???")
    w_name = next(w["name"] for w in YOUR_WORKFLOWS if w["id"] == wid)
    if yours == "???":
        print(f"  {wid}. {w_name:<45} -- NOT ANSWERED")
        all_correct = False
    elif yours == correct:
        print(f"  {wid}. {w_name:<45} {yours:<8} CORRECT")
    else:
        print(f"  {wid}. {w_name:<45} {yours:<8} WRONG (should be {correct})")
        reason = "Blocking workflow -- humans/processes are waiting" if correct == "sync" else "Non-blocking -- no latency requirement"
        print(f"     Reason: {reason}")
        all_correct = False

if all_correct and "???" not in your_classifications.values():
    print("\n  All 8 correct!")

print(f"\n=== TASK 2: Submission Frequency Check ===")
correct_interval = 6  # 30 - 24 = 6 hours
if your_submission_interval == "???":
    print(f"  Not answered yet.")
    print(f"  Hint: max_interval = sla_hours - batch_max_processing_time")
elif your_submission_interval == correct_interval or str(your_submission_interval) == str(correct_interval):
    print(f"  Correct! 30h SLA - 24h max processing = submit every 6 hours.")
else:
    print(f"  Your answer: {your_submission_interval}h. Correct: {correct_interval}h")
    print(f"  Formula: {sla_hours}h SLA - 24h max batch time = {correct_interval}h max interval")

print(f"\n=== TASK 3: Failure Recovery Check ===")
if your_failure_plan["expected_failures"] == "???":
    print("  Not answered yet.")
    print("  Key points to address:")
    print("    - 200 docs * 5% = 10 expected failures")
    print("    - Resubmit ONLY the 10 failed (use custom_id to identify them)")
    print("    - Do NOT resubmit the 190 that succeeded")
    print("    - custom_id is how you match failures to original requests")
else:
    print(f"  Expected failures: {your_failure_plan['expected_failures']}")
    expected = 10  # 200 * 0.05
    if str(your_failure_plan['expected_failures']) == str(expected):
        print(f"    Correct! 200 * 5% = 10")
    else:
        print(f"    Expected: {expected} (200 * 5%)")
    print(f"  Resubmit: {your_failure_plan['resubmit_what']}")
    print(f"  Skip: {your_failure_plan['do_not_resubmit']}")
    print(f"  Identification method: {your_failure_plan['how_to_identify_failures']}")

---
## Phase 4: STRESS TEST -- Edge Cases

The exam loves edge cases. Let's work through the tricky ones.

In [ ]:
# === EDGE CASE 1: "Usually non-blocking but sometimes urgent" ===

edge_case_1 = {
    "name": "Documentation generation",
    "description": (
        "Normally runs overnight to update docs. But sometimes a major release "
        "needs docs generated within 2 hours for a client demo."
    ),
    "usually_blocking": False,
    "sometimes_blocking": True,
}

print("=== EDGE CASE 1: Sometimes-Urgent Workflow ===")
print(f"\n  Scenario: {edge_case_1['description']}")
print(f"\n  Wrong answer: Use batch (it's usually non-blocking)")
print(f"  Wrong answer: Use sync (it's usually non-blocking, waste of money)")
print(f"\n  Correct answer: Use SYNC")
print(f"  Why: If it EVER needs fast results, you cannot rely on batch.")
print(f"  Batch has no latency SLA. When the client demo happens and batch")
print(f"  takes 20 hours, you've failed. Use sync and accept the cost.")
print(f"\n  Alternative: Build BOTH paths -- batch for normal, sync for urgent.")
print(f"  But on the exam, if forced to pick one: sync.")

In [ ]:
# === EDGE CASE 2: SLA of exactly 24 hours ===

print("=== EDGE CASE 2: 24-Hour SLA ===")
print()

sla = 24
interval = calculate_submission_interval(sla)

print(f"  SLA: {sla} hours")
print(f"  Batch max processing: 24 hours")
print(f"  Buffer: {sla} - 24 = 0 hours")
print(f"  Submission interval: {interval}")
print()
print(f"  This means: You'd need to submit continuously (every 0 hours),")
print(f"  which is impossible. Batch CANNOT guarantee a 24-hour SLA.")
print()
print(f"  On the exam: A 24-hour SLA requires synchronous API.")
print(f"  Batch only works when SLA is STRICTLY greater than 24 hours.")

In [ ]:
# === EDGE CASE 3: 100% failure rate ===

print("=== EDGE CASE 3: 100% Batch Failure ===")
print()

# Simulate a batch where everything fails
all_fail_results = simulate_batch_results(batch_requests, failure_rate=1.0)
failed_count = sum(1 for r in all_fail_results if r["status"] == "errored")

print(f"  Submitted: {len(batch_requests)} requests")
print(f"  Failed:    {failed_count} (100%)")
print()
print(f"  Wrong reaction: Retry the entire batch")
print(f"  Wrong reaction: Retry 3 times then give up")
print()
print(f"  Correct reaction: INVESTIGATE THE PROMPT")
print(f"  When 100% of documents fail, the issue isn't transient.")
print(f"  Possible causes:")
print(f"    - Documents exceed context window (need chunking)")
print(f"    - Prompt is malformed")
print(f"    - Document format is unexpected (binary, encoding issues)")
print(f"    - API permissions or quota issue")
print()
print(f"  Retrying a fundamentally broken batch wastes time and money.")
print(f"  Fix the root cause first.")

In [ ]:
# === EDGE CASE 4: Batch API limitations ===

print("=== EDGE CASE 4: What Batch API Cannot Do ===")
print()
print("  The Batch API does NOT support multi-turn tool calling.")
print("  This means you cannot:")
print("    - Have Claude call a tool and get results back mid-request")
print("    - Run iterative agent loops (call tool -> analyze -> call tool)")
print("    - Use real-time function execution")
print()
print("  If your workflow requires tool use with real-time execution:")
print("    -> You MUST use the synchronous API")
print()
print("  Batch CAN include tool definitions in the request, but cannot")
print("  execute tools and return results during processing.")

---
## Key Takeaways

1. **Batch = 50% savings, but no latency SLA.** Up to 24 hours, no guarantees.
2. **Blocking workflows MUST use sync.** "Usually fast" is not an SLA.
3. **SLA math:** `max_interval = sla_hours - 24`. If zero or negative, batch is impossible.
4. **`custom_id` is your lifeline.** Use it to correlate requests, identify failures, resubmit only what failed.
5. **Smart failure handling:** Retry transient errors, chunk oversized docs, investigate 100% failures.
6. **No multi-turn tool calling** in batch -- use sync for agent workflows.

## Next

Lab 1.6: Multi-Instance and Multi-Pass Review